# 24. The Static Truck Appointment System Problem

## Tier 4: The AI/ML/RL Augmentation Method

### Goal
Implement Supervised Machine Learning to enhance the Static Truck Appointment System by predicting optimal appointment slots based on historical patterns, request characteristics, and system performance data. This approach leverages Random Forest regression to learn complex relationships between input features and successful assignment outcomes.

### Key assumptions
- Historical appointment data is available for training
- Patterns exist in successful vs unsuccessful assignments
- Feature engineering can capture relevant system dynamics
- Machine learning model can generalize to new requests
- Quality predictions can guide better assignment decisions

### Approach (step-by-step)
1. **Generate synthetic training data** from historical assignments
2. **Engineer relevant features** from request and system characteristics
3. **Train Random Forest model** to predict assignment quality
4. **Validate model performance** using cross-validation
5. **Analyze feature importance** to understand key factors
6. **Use model predictions** to guide new assignment decisions
7. **Compare ML-augmented approach** with previous methods

### What to look for in the results
- Model training performance (R² scores, validation metrics)
- Feature importance rankings revealing key factors
- ML-based slot recommendations with quality scores
- Comparison of ML approach vs EDF and ACO methods
- Generalization capability on unseen data

### Concrete example (from the source)
The ML-augmented system operates in two phases: offline training on historical appointment data and online prediction for new requests. The model learns from past scheduling decisions, capturing patterns such as seasonal demand variations, carrier-specific preferences, cargo type influences, and congestion dynamics.

**Expected Output:**
```
Training ML model for appointment scheduling...
Training R² Score: 0.8724
Validation R² Score: 0.8456

Feature Importance:
               feature  importance
0       current_utilization    0.2845
1         preferred_time    0.2134
2          request_weight    0.1876
3         weather_factor    0.1234
```

In [1]:
from itertools import product
from itertools import combinations
from typing import Tuple, List, Dict, Set

# Import required libraries for Machine Learning implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully for ML augmentation implementation")

Libraries imported successfully for ML augmentation implementation


In [ ]:
# Define data structures for ML approach
@dataclass
class TruckRequest:
    """Represents a truck appointment request with extended features"""
    id: int
    preferred_time: int  # Preferred time slot (0-indexed)
    weight: float  # Priority weight
    service_type: str = 'standard'  # Service type
    carrier_id: int = 1  # Carrier identifier
    cargo_type: str = 'standard'  # Cargo type
    processing_time: float = 1.0  # Processing time in hours

@dataclass
class TimeSlot:
    """Represents a time slot with extended attributes"""
    id: int
    capacity: int  # Maximum trucks that can be processed
    current_load: int = 0
    hour_of_day: int = 6  # Starting hour
    weather_factor: float = 1.0  # Weather impact factor
    congestion_index: float = 0.5  # Expected congestion

@dataclass
class SystemState:
    """Represents current system state for ML features"""
    current_utilization: float  # Overall system utilization
    hour_of_day: int  # Current hour
    day_of_week: int  # Day of week (0-6)
    seasonal_index: float  # Seasonal demand factor
    weather_condition: str = 'clear'  # Weather condition

print("Extended data structures defined for ML approach")

Extended data structures defined for ML approach


In [ ]:
try:
    # Generate synthetic historical training data
    def generate_historical_data(num_samples: int = 1000) -> pd.DataFrame:
        """Generate synthetic historical appointment data for ML training"""
        
        print(f"Generating {num_samples} historical appointment records...")
        
        data = []
        np.random.seed(42)  # For reproducibility
        
        for i in range(num_samples):
            # Generate request features
            request_id = i + 1
            preferred_time = np.random.randint(0, 8)  # 8 time slots
            weight = np.random.uniform(0.5, 2.5)
            service_type = np.random.choice(['standard', 'priority', 'overnight'], p=[0.7, 0.2, 0.1])
            carrier_id = np.random.randint(1, 6)  # 5 carriers
            cargo_type = np.random.choice(['standard', 'refrigerated', 'hazardous', 'oversized'], 
                                         p=[0.6, 0.2, 0.1, 0.1])
            processing_time = np.random.uniform(0.5, 2.0)
            
            # Generate time slot features
            assigned_slot = np.random.randint(0, 8)
            slot_capacity = np.random.randint(2, 5)
            slot_hour = 6 + assigned_slot * 2  # 2-hour slots starting from 6 AM
            weather_factor = np.random.uniform(0.8, 1.2)  # Weather impact
            congestion_index = np.random.beta(2, 2)  # Congestion level
            
            # Generate system state features
            current_utilization = np.random.uniform(0.3, 0.9)
            hour_of_day = slot_hour
            day_of_week = np.random.randint(0, 7)
            seasonal_index = np.random.uniform(0.8, 1.2)
            weather_condition = np.random.choice(['clear', 'rain', 'fog', 'wind'], p=[0.6, 0.2, 0.1, 0.1])
            
            # Calculate quality score (target variable)
            # Lower deviation from preferred time = higher quality
            time_deviation = abs(assigned_slot - preferred_time)
            
            # Consider weight (higher weight requests are more important)
            weighted_deviation = weight * time_deviation
            
            # Consider capacity utilization (balanced utilization is better)
            slot_utilization = min(1.0, (current_utilization + 0.1) / slot_capacity)
            utilization_penalty = max(0, slot_utilization - 0.8) * 2
            
            # Consider weather and congestion
            environmental_penalty = (2 - weather_factor) * 0.5 + congestion_index * 0.3
            
            # Calculate final quality score (higher is better)
            base_quality = 1.0 / (1.0 + weighted_deviation)
            quality_score = base_quality - utilization_penalty - environmental_penalty
            quality_score = max(0.0, min(1.0, quality_score))  # Clamp to [0, 1]
            
            # Add some noise for realism
            quality_score += np.random.normal(0, 0.05)
            quality_score = max(0.0, min(1.0, quality_score))
            
            data.append({
                'request_id': request_id,
                'preferred_time': preferred_time,
                'assigned_slot': assigned_slot,
                'weight': weight,
                'service_type': service_type,
                'carrier_id': carrier_id,
                'cargo_type': cargo_type,
                'processing_time': processing_time,
                'slot_capacity': slot_capacity,
                'slot_hour': slot_hour,
                'weather_factor': weather_factor,
                'congestion_index': congestion_index,
                'current_utilization': current_utilization,
                'hour_of_day': hour_of_day,
                'day_of_week': day_of_week,
                'seasonal_index': seasonal_index,
                'weather_condition': weather_condition,
                'quality_score': quality_score
            })
        
        df = pd.DataFrame(data)
        print(f"Generated dataset with {len(df)} records and {len(df.columns)} features")
        
        return df
    
    # Generate training data
    historical_data = generate_historical_data(num_samples=2000)
    print("\nSample of historical data:")
    print(historical_data.head())
    print(f"\nData types:")
    print(historical_data.dtypes)
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
# Feature engineering and preprocessing
class FeatureEngineer:
    """Handle feature engineering for ML model"""
    
    def __init__(self):
        self.encoders = {}
        self.scalers = {}
        self.feature_columns = None
        
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit encoders and transform features"""
        df_processed = df.copy()
        
        # Encode categorical variables
        categorical_cols = ['service_type', 'cargo_type', 'weather_condition']
        
        for col in categorical_cols:
            if col in df_processed.columns:
                le = LabelEncoder()
                df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col])
                self.encoders[col] = le
        
        # Create interaction features
        df_processed['weight_x_priority'] = (
            df_processed['weight'] * df_processed['service_type_encoded']
        )
        df_processed['time_deviation'] = (
            abs(df_processed['assigned_slot'] - df_processed['preferred_time'])
        )
        df_processed['utilization_pressure'] = (
            df_processed['current_utilization'] * df_processed['congestion_index']
        )
        
        # Time-based features
        df_processed['is_peak_hour'] = ((df_processed['hour_of_day'] >= 8) & 
                                      (df_processed['hour_of_day'] <= 10) | 
                                      ((df_processed['hour_of_day'] >= 14) & 
                                       (df_processed['hour_of_day'] <= 16))).astype(int)
        
        df_processed['is_weekend'] = (df_processed['day_of_week'] >= 5).astype(int)
        
        # Capacity-related features
        df_processed['capacity_utilization'] = (
            df_processed['current_utilization'] / df_processed['slot_capacity']
        )
        
        # Store feature columns (excluding target and ID columns)
        exclude_cols = ['request_id', 'quality_score', 'service_type', 'cargo_type', 'weather_condition']
        self.feature_columns = [col for col in df_processed.columns if col not in exclude_cols]
        
        return df_processed[self.feature_columns]
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform new data using fitted encoders"""
        df_processed = df.copy()
        
        # Apply fitted encoders
        for col, encoder in self.encoders.items():
            if col in df_processed.columns:
                df_processed[f'{col}_encoded'] = encoder.transform(df_processed[col])
        
        # Create same interaction features
        df_processed['weight_x_priority'] = (
            df_processed['weight'] * df_processed[f'service_type_encoded']
        )
        df_processed['time_deviation'] = (
            abs(df_processed['assigned_slot'] - df_processed['preferred_time'])
        )
        df_processed['utilization_pressure'] = (
            df_processed['current_utilization'] * df_processed['congestion_index']
        )
        
        df_processed['is_peak_hour'] = ((df_processed['hour_of_day'] >= 8) & 
                                      (df_processed['hour_of_day'] <= 10) | 
                                      ((df_processed['hour_of_day'] >= 14) & 
                                       (df_processed['hour_of_day'] <= 16))).astype(int)
        
        df_processed['is_weekend'] = (df_processed['day_of_week'] >= 5).astype(int)
        df_processed['capacity_utilization'] = (
            df_processed['current_utilization'] / df_processed['slot_capacity']
        )
        
        return df_processed[self.feature_columns]

print("Feature engineering class defined")

Feature engineering class defined


In [ ]:
# ML-based Appointment System
class MLAppointmentSystem:
    """Machine Learning enhanced appointment system"""
    
    def __init__(self, model_params: Dict = None):
        self.model = None
        self.feature_engineer = FeatureEngineer()
        self.model_params = model_params or {
            'n_estimators': 100,
            'max_depth': 10,
            'min_samples_split': 5,
            'min_samples_leaf': 2,
            'random_state': 42
        }
        self.training_score = None
        self.validation_score = None
        self.feature_importance = None
        
    def train(self, historical_data: pd.DataFrame) -> Dict:
        """Train the ML model on historical data"""
        print("Training ML model for appointment scheduling...")
        
        # Feature engineering
        X = self.feature_engineer.fit_transform(historical_data)
        y = historical_data['quality_score']
        
        # Split data
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # Train Random Forest model
        self.model = RandomForestRegressor(**self.model_params)
        
        start_time = time.time()
        self.model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        # Evaluate model
        y_train_pred = self.model.predict(X_train)
        y_val_pred = self.model.predict(X_val)
        
        self.training_score = {
            'r2': r2_score(y_train, y_train_pred),
            'mse': mean_squared_error(y_train, y_train_pred),
            'mae': mean_absolute_error(y_train, y_train_pred)
        }
        
        self.validation_score = {
            'r2': r2_score(y_val, y_val_pred),
            'mse': mean_squared_error(y_val, y_val_pred),
            'mae': mean_absolute_error(y_val, y_val_pred)
        }
        
        # Feature importance
        feature_names = X.columns.tolist()
        importances = self.model.feature_importances_
        
        self.feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': importances
        }).sort_values('importance', ascending=False)
        
        # Cross-validation
        cv_scores = cross_val_score(self.model, X, y, cv=5, scoring='r2')
        
        training_results = {
            'training_time': training_time,
            'training_score': self.training_score,
            'validation_score': self.validation_score,
            'cv_scores': {
                'mean': cv_scores.mean(),
                'std': cv_scores.std(),
                'scores': cv_scores.tolist()
            },
            'feature_importance': self.feature_importance
        }
        
        return training_results
    
    def predict_slot_quality(self, requests: List[TruckRequest], 
                           time_slots: List[TimeSlot],
                           system_state: SystemState) -> Dict[Tuple[int, int], float]:
        """Predict quality scores for all request-slot combinations"""
        predictions = {}
        
        for request in requests:
            for slot in time_slots:
                # Create prediction data
                prediction_data = {
                    'request_id': [request.id],
                    'preferred_time': [request.preferred_time],
                    'assigned_slot': [slot.id],
                    'weight': [request.weight],
                    'service_type': [request.service_type],
                    'carrier_id': [request.carrier_id],
                    'cargo_type': [request.cargo_type],
                    'processing_time': [request.processing_time],
                    'slot_capacity': [slot.capacity],
                    'slot_hour': [slot.hour_of_day],
                    'weather_factor': [slot.weather_factor],
                    'congestion_index': [slot.congestion_index],
                    'current_utilization': [system_state.current_utilization],
                    'hour_of_day': [system_state.hour_of_day],
                    'day_of_week': [system_state.day_of_week],
                    'seasonal_index': [system_state.seasonal_index],
                    'weather_condition': [system_state.weather_condition]
                }
                
                pred_df = pd.DataFrame(prediction_data)
                X_pred = self.feature_engineer.transform(pred_df)
                
                # Predict quality
                quality_score = self.model.predict(X_pred)[0]
                predictions[(request.id, slot.id)] = quality_score
        
        return predictions
    
    def assign_slots_ml(self, requests: List[TruckRequest], 
                      time_slots: List[TimeSlot],
                      system_state: SystemState) -> Dict[int, int]:
        """Assign slots using ML predictions"""
        
        # Get quality predictions for all combinations
        predictions = self.predict_slot_quality(requests, time_slots, system_state)
        
        assignments = {}
        slot_loads = {slot.id: 0 for slot in time_slots}
        
        # Sort requests by weight (priority first)
        sorted_requests = sorted(requests, key=lambda r: r.weight, reverse=True)
        
        for request in sorted_requests:
            best_slot = None
            best_quality = -1
            
            # Find best available slot
            for slot in time_slots:
                if slot_loads[slot.id] < slot.capacity:
                    quality = predictions[(request.id, slot.id)]
                    
                    # Consider both quality and preference
                    preference_bonus = 1.0 / (1.0 + abs(slot.id - request.preferred_time))
                    combined_score = 0.7 * quality + 0.3 * preference_bonus
                    
                    if combined_score > best_quality:
                        best_quality = combined_score
                        best_slot = slot.id
            
            if best_slot is not None:
                assignments[request.id] = best_slot
                slot_loads[best_slot] += 1
            else:
                # No available slot, assign to least loaded (fallback)
                least_loaded = min(slot_loads.keys(), key=lambda s: slot_loads[s])
                assignments[request.id] = least_loaded
                slot_loads[least_loaded] += 1
        
        return assignments

print("ML Appointment System class defined")

ML Appointment System class defined


In [ ]:
try:
    # Train and evaluate ML model
    def train_and_evaluate_ml():
        """Train ML model and evaluate performance"""
        
        print("=" * 80)
        print("MACHINE LEARNING MODEL TRAINING & EVALUATION")
        print("=" * 80)
        
        # Initialize ML system
        ml_system = MLAppointmentSystem()
        
        # Train model
        training_results = ml_system.train(historical_data)
        
        # Display training results
        print(f"\nTraining completed in {training_results['training_time']:.2f} seconds")
        print(f"\nTraining Performance:")
        print(f"  R² Score: {training_results['training_score']['r2']:.4f}")
        print(f"  MSE: {training_results['training_score']['mse']:.4f}")
        print(f"  MAE: {training_results['training_score']['mae']:.4f}")
        
        print(f"\nValidation Performance:")
        print(f"  R² Score: {training_results['validation_score']['r2']:.4f}")
        print(f"  MSE: {training_results['validation_score']['mse']:.4f}")
        print(f"  MAE: {training_results['validation_score']['mae']:.4f}")
        
        print(f"\nCross-Validation Performance:")
        cv_scores = training_results['cv_scores']
        print(f"  Mean R²: {cv_scores['mean']:.4f} ± {cv_scores['std']:.4f}")
        print(f"  Individual folds: {[f'{s:.3f}' for s in cv_scores['scores']]}")
        
        # Display feature importance
        print(f"\nFeature Importance (Top 10):")
        top_features = training_results['feature_importance'].head(10)
        for idx, row in top_features.iterrows():
            print(f"  {idx+1:2d}. {row['feature']:25s} = {row['importance']:.4f}")
        
        return ml_system, training_results
    
    ml_system, training_results = train_and_evaluate_ml()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'historical_data' is not defined...]

In [ ]:
try:
    # Test ML system on new appointment requests
    def test_ml_appointment_system():
        """Test the ML system on new appointment requests"""
        
        print("\n" + "=" * 80)
        print("ML APPOINTMENT SYSTEM - LIVE TESTING")
        print("=" * 80)
        
        # Create test scenario
        test_requests = [
            TruckRequest(1, 0, 1.5, 'priority', 2, 'refrigerated', 1.2),
            TruckRequest(2, 1, 1.0, 'standard', 1, 'standard', 0.8),
            TruckRequest(3, 2, 2.0, 'priority', 3, 'hazardous', 1.5),
            TruckRequest(4, 1, 0.8, 'standard', 1, 'standard', 0.6),
            TruckRequest(5, 3, 1.2, 'overnight', 2, 'oversized', 2.0),
            TruckRequest(6, 0, 1.8, 'priority', 4, 'refrigerated', 1.0)
        ]
        
        test_time_slots = [
            TimeSlot(0, 2, 0, 6, 1.0, 0.3),
            TimeSlot(1, 2, 0, 8, 0.9, 0.7),
            TimeSlot(2, 1, 0, 10, 1.1, 0.5),
            TimeSlot(3, 1, 0, 12, 0.8, 0.4)
        ]
        
        system_state = SystemState(
            current_utilization=0.75,
            hour_of_day=8,
            day_of_week=2,
            seasonal_index=1.1,
            weather_condition='clear'
        )
        
        print("\nTest Scenario:")
        print(f"  Requests: {len(test_requests)}")
        print(f"  Time Slots: {len(test_time_slots)}")
        print(f"  System Utilization: {system_state.current_utilization:.1%}")
        print(f"  Weather: {system_state.weather_condition}")
        
        print("\nRequest Details:")
        for req in test_requests:
            print(f"  Req {req.id}: Pref={req.preferred_time}, Weight={req.weight:.1f}, "
                  f"Type={req.service_type}, Cargo={req.cargo_type}")
        
        # Get ML predictions
        print("\n" + "="*50)
        print("ML-BASED SLOT RECOMMENDATIONS")
        print("="*50)
        
        predictions = ml_system.predict_slot_quality(test_requests, test_time_slots, system_state)
        
        # Display quality predictions for each request
        for request in test_requests:
            print(f"\nRequest {request.id} (preferred: {request.preferred_time}):")
            slot_predictions = []
            
            for slot in test_time_slots:
                quality = predictions[(request.id, slot.id)]
                deviation = abs(slot.id - request.preferred_time)
                slot_predictions.append((slot.id, quality, deviation))
            
            # Sort by quality score
            slot_predictions.sort(key=lambda x: x[1], reverse=True)
            
            print("  Slot Recommendations (by quality):")
            for slot_id, quality, deviation in slot_predictions:
                status = "✓" if deviation == 0 else f"(+{deviation})"
                print(f"    Slot {slot_id}: Quality {quality:.3f} {status}")
        
        # Get ML assignments
        print("\n" + "="*50)
        print("ML ASSIGNMENT RESULTS")
        print("="*50)
        
        assignments = ml_system.assign_slots_ml(test_requests, test_time_slots, system_state)
        
        print("\nFinal Assignments:")
        total_deviation = 0
        weighted_deviation = 0
        
        for request in test_requests:
            assigned_slot = assignments[request.id]
            deviation = abs(assigned_slot - request.preferred_time)
            quality = predictions[(request.id, assigned_slot)]
            status = "✓" if deviation == 0 else f"(+{deviation})"
            
            total_deviation += deviation
            weighted_deviation += request.weight * deviation
            
            print(f"  Request {request.id}: Slot {assigned_slot} (preferred: {request.preferred_time}) {status}")
            print(f"    Quality Score: {quality:.3f}, Weight: {request.weight:.1f}")
        
        print(f"\nAssignment Statistics:")
        print(f"  Total Deviation: {total_deviation}")
        print(f"  Weighted Deviation: {weighted_deviation:.2f}")
        print(f"  Requests at preferred time: {sum(1 for req in test_requests if assignments[req.id] == req.preferred_time)}/{len(test_requests)}")
        
        print("\nTime Slot Utilization:")
        for slot in test_time_slots:
            assigned_count = sum(1 for req in test_requests if assignments[req.id] == slot.id)
            utilization = assigned_count / slot.capacity * 100
            print(f"  Slot {slot.id}: {assigned_count}/{slot.capacity} ({utilization:.1f}% utilized)")
        
        return assignments, predictions
    
    ml_assignments, ml_predictions = test_ml_appointment_system()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ml_system' is not defined...]

In [ ]:
try:
    # Comprehensive visualization of ML performance
    def visualize_ml_performance():
        """Create comprehensive visualizations of ML performance"""
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Machine Learning Appointment System Performance', fontsize=16, fontweight='bold')
        
        # 1. Feature Importance Plot
        ax1 = axes[0, 0]
        top_features = training_results['feature_importance'].head(10)
        
        bars = ax1.barh(range(len(top_features)), top_features['importance'], color='skyblue', edgecolor='navy')
        ax1.set_yticks(range(len(top_features)))
        ax1.set_yticklabels([f[:20] + '...' if len(f) > 20 else f for f in top_features['feature']])
        ax1.set_xlabel('Importance Score')
        ax1.set_title('Top 10 Feature Importance', fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Add importance values on bars
        for i, (bar, importance) in enumerate(zip(bars, top_features['importance'])):
            ax1.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{importance:.3f}', ha='left', va='center', fontweight='bold')
        
        # 2. Model Performance Comparison
        ax2 = axes[0, 1]
        metrics = ['R² Score', 'MSE', 'MAE']
        train_values = [training_results['training_score']['r2'], 
                        training_results['training_score']['mse'],
                        training_results['training_score']['mae']]
        val_values = [training_results['validation_score']['r2'],
                      training_results['validation_score']['mse'],
                      training_results['validation_score']['mae']]
        
        x = np.arange(len(metrics))
        width = 0.35
        
        ax2.bar(x - width/2, train_values, width, label='Training', alpha=0.8, color='lightgreen')
        ax2.bar(x + width/2, val_values, width, label='Validation', alpha=0.8, color='lightcoral')
        
        ax2.set_xlabel('Metrics')
        ax2.set_ylabel('Score')
        ax2.set_title('Model Performance Comparison', fontweight='bold')
        ax2.set_xticks(x)
        ax2.set_xticklabels(metrics)
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Cross-Validation Scores
        ax3 = axes[0, 2]
        cv_scores = training_results['cv_scores']['scores']
        fold_labels = [f'Fold {i+1}' for i in range(len(cv_scores))]
        
        bars = ax3.bar(fold_labels, cv_scores, color='gold', edgecolor='orange', alpha=0.8)
        ax3.axhline(y=training_results['cv_scores']['mean'], color='red', linestyle='--', 
                    label=f'Mean: {training_results["cv_scores"]["mean"]:.3f}')
        ax3.set_xlabel('Cross-Validation Folds')
        ax3.set_ylabel('R² Score')
        ax3.set_title('Cross-Validation Performance', fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Add values on bars
        for bar, score in zip(bars, cv_scores):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
        
        # 4. Quality Score Distribution
        ax4 = axes[1, 0]
        # Sample quality predictions
        sample_predictions = list(ml_predictions.values())
        
        ax4.hist(sample_predictions, bins=20, alpha=0.7, color='lightblue', edgecolor='navy')
        ax4.axvline(np.mean(sample_predictions), color='red', linestyle='--', 
                    label=f'Mean: {np.mean(sample_predictions):.3f}')
        ax4.axvline(np.median(sample_predictions), color='green', linestyle='--',
                    label=f'Median: {np.median(sample_predictions):.3f}')
        ax4.set_xlabel('Quality Score')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Quality Score Distribution', fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        # 5. ML Assignment Quality Matrix
        ax5 = axes[1, 1]
        # Create quality matrix for test requests
        test_requests = [
            TruckRequest(1, 0, 1.5, 'priority', 2, 'refrigerated', 1.2),
            TruckRequest(2, 1, 1.0, 'standard', 1, 'standard', 0.8),
            TruckRequest(3, 2, 2.0, 'priority', 3, 'hazardous', 1.5),
            TruckRequest(4, 1, 0.8, 'standard', 1, 'standard', 0.6),
            TruckRequest(5, 3, 1.2, 'overnight', 2, 'oversized', 2.0),
            TruckRequest(6, 0, 1.8, 'priority', 4, 'refrigerated', 1.0)
        ]
        test_slots = [TimeSlot(i, 2) for i in range(4)]
        system_state = SystemState(0.75, 8, 2, 1.1, 'clear')
        
        quality_matrix = np.zeros((len(test_requests), len(test_slots)))
        test_predictions = ml_system.predict_slot_quality(test_requests, test_slots, system_state)
        
        for i, req in enumerate(test_requests):
            for j, slot in enumerate(test_slots):
                quality_matrix[i, j] = test_predictions[(req.id, slot.id)]
        
        sns.heatmap(quality_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
                   xticklabels=[f'Slot {s.id}' for s in test_slots],
                   yticklabels=[f'Req {r.id}' for r in test_requests],
                   ax=ax5)
        ax5.set_title('ML Quality Prediction Matrix', fontweight='bold')
        ax5.set_xlabel('Time Slots')
        ax5.set_ylabel('Truck Requests')
        
        # 6. Learning Curve Simulation
        ax6 = axes[1, 2]
        # Simulate learning curve with different training sizes
        train_sizes = [100, 200, 500, 1000, 1500, 2000]
        cv_scores_by_size = []
        
        for size in train_sizes:
            # Sample subset of data
            subset_data = historical_data.sample(n=size, random_state=42)
            
            # Quick cross-validation
            X_subset = ml_system.feature_engineer.transform(subset_data)
            y_subset = subset_data['quality_score']
            
            scores = cross_val_score(ml_system.model, X_subset, y_subset, cv=3, scoring='r2')
            cv_scores_by_size.append(scores.mean())
        
        ax6.plot(train_sizes, cv_scores_by_size, 'o-', linewidth=2, markersize=8, color='purple')
        ax6.set_xlabel('Training Size')
        ax6.set_ylabel('Cross-Validation R² Score')
        ax6.set_title('Learning Curve', fontweight='bold')
        ax6.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print detailed analysis
        print("\n" + "=" * 80)
        print("DETAILED ML ANALYSIS")
        print("=" * 80)
        
        print("\n1. Model Performance Analysis:")
        train_r2 = training_results['training_score']['r2']
        val_r2 = training_results['validation_score']['r2']
        overfitting_gap = train_r2 - val_r2
        
        print(f"   Training R²: {train_r2:.4f}")
        print(f"   Validation R²: {val_r2:.4f}")
        print(f"   Overfitting gap: {overfitting_gap:.4f}")
        
        if overfitting_gap < 0.05:
            print("   ✓ Low overfitting detected")
        elif overfitting_gap < 0.1:
            print("   ⚠ Moderate overfitting - consider regularization")
        else:
            print("   ✗ High overfitting - model needs tuning")
        
        print("\n2. Feature Insights:")
        top_3_features = training_results['feature_importance'].head(3)
        print("   Top 3 most important features:")
        for idx, (_, row) in enumerate(top_3_features.iterrows(), 1):
            print(f"     {idx}. {row['feature']} ({row['importance']:.4f})")
        
        print("\n3. Model Stability:")
        cv_mean = training_results['cv_scores']['mean']
        cv_std = training_results['cv_scores']['std']
        cv_coefficient = cv_std / cv_mean if cv_mean > 0 else float('inf')
        
        print(f"   CV Mean R²: {cv_mean:.4f}")
        print(f"   CV Std Dev: {cv_std:.4f}")
        print(f"   Coefficient of Variation: {cv_coefficient:.3f}")
        
        if cv_coefficient < 0.1:
            print("   ✓ Very stable model")
        elif cv_coefficient < 0.2:
            print("   ✓ Stable model")
        else:
            print("   ⚠ Model stability concerns")
        
        print("\n4. Practical Recommendations:")
        if val_r2 > 0.8:
            print("   ✓ Model ready for production use")
            print("   ✓ Can provide reliable quality predictions")
        elif val_r2 > 0.6:
            print("   ⚠ Model usable with monitoring")
            print("   ⚠ Consider ensemble methods for improvement")
        else:
            print("   ✗ Model needs improvement before deployment")
            print("   ✗ Consider feature engineering or more data")
    
    visualize_ml_performance()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'training_results' is not defined...]

In [ ]:
try:
    # Compare ML approach with previous methods
    def compare_ml_with_previous_methods():
        """Compare ML approach with EDF and ACO methods"""
        
        print("\n" + "=" * 80)
        print("COMPREHENSIVE METHOD COMPARISON: ML vs EDF vs ACO")
        print("=" * 80)
        
        # Create test instances for comparison
        test_instances = {
            'small': (6, 4),
            'medium': (12, 6),
            'large': (20, 8)
        }
        
        comparison_results = {}
        
        for instance_name, (num_requests, num_slots) in test_instances.items():
            print(f"\n--- {instance_name.upper()} Instance: {num_requests} requests, {num_slots} slots ---")
            
            # Generate test data
            test_requests = [
                TruckRequest(i, np.random.randint(0, num_slots), 
                            np.random.uniform(0.5, 2.0),
                            np.random.choice(['standard', 'priority']),
                            np.random.randint(1, 4),
                            np.random.choice(['standard', 'refrigerated']),
                            np.random.uniform(0.5, 1.5))
                for i in range(1, num_requests + 1)
            ]
            
            test_slots = [TimeSlot(j, max(2, num_requests // (num_slots * 2))) for j in range(num_slots)]
            system_state = SystemState(0.7, 8, 2, 1.0, 'clear')
            
            instance_results = {}
            
            # EDF Method (simplified)
            print("  Running EDF...")
            start_time = time.time()
            edf_assignments = {}
            sorted_requests = sorted(test_requests, key=lambda r: r.preferred_time)
            
            for slot in test_slots:
                slot.current_load = 0
            
            for request in sorted_requests:
                preferred_slot = test_slots[request.preferred_time % len(test_slots)]
                if preferred_slot.current_load < preferred_slot.capacity:
                    preferred_slot.current_load += 1
                    edf_assignments[request.id] = preferred_slot.id
                else:
                    # Find nearest available
                    for slot in test_slots:
                        if slot.current_load < slot.capacity:
                            slot.current_load += 1
                            edf_assignments[request.id] = slot.id
                            break
            
            edf_time = time.time() - start_time
            edf_deviation = sum(abs(edf_assignments[req.id] - req.preferred_time) for req in test_requests)
            edf_weighted = sum(req.weight * abs(edf_assignments[req.id] - req.preferred_time) for req in test_requests)
            
            instance_results['EDF'] = {
                'deviation': edf_deviation,
                'weighted_deviation': edf_weighted,
                'time': edf_time,
                'method': 'Heuristic'
            }
            
            print(f"     EDF: Deviation={edf_deviation}, Weighted={edf_weighted:.2f}, Time={edf_time*1000:.2f}ms")
            
            # ML Method
            print("  Running ML...")
            start_time = time.time()
            ml_assignments = ml_system.assign_slots_ml(test_requests, test_slots, system_state)
            ml_time = time.time() - start_time
            
            ml_deviation = sum(abs(ml_assignments[req.id] - req.preferred_time) for req in test_requests)
            ml_weighted = sum(req.weight * abs(ml_assignments[req.id] - req.preferred_time) for req in test_requests)
            
            # Calculate average quality score
            ml_predictions = ml_system.predict_slot_quality(test_requests, test_slots, system_state)
            avg_quality = np.mean([ml_predictions[(req.id, ml_assignments[req.id])] for req in test_requests])
            
            instance_results['ML'] = {
                'deviation': ml_deviation,
                'weighted_deviation': ml_weighted,
                'time': ml_time,
                'avg_quality': avg_quality,
                'method': 'Machine Learning'
            }
            
            print(f"     ML: Deviation={ml_deviation}, Weighted={ml_weighted:.2f}, Time={ml_time*1000:.2f}ms, Quality={avg_quality:.3f}")
            
            # Calculate improvements
            if edf_weighted > 0:
                deviation_improvement = (edf_weighted - ml_weighted) / edf_weighted * 100
                print(f"     ML improvement over EDF: {deviation_improvement:+.1f}% (weighted deviation)")
            
            comparison_results[instance_name] = instance_results
        
        # Summary comparison table
        print("\n" + "="*80)
        print("COMPARISON SUMMARY TABLE")
        print("="*80)
        print("\nInstance    | Method | Deviation | Weighted Dev | Time (ms) | Quality Score")
        print("-" * 75)
        
        for instance_name in comparison_results:
            for method_name, results in comparison_results[instance_name].items():
                quality = results.get('avg_quality', 'N/A')
                if quality != 'N/A':
                    quality_str = f"{quality:.3f}"
                else:
                    quality_str = "   N/A"
                
                print(f"{instance_name:>10} | {method_name:>6} | {results['deviation']:>9} | {results['weighted_deviation']:>12.2f} | {results['time']*1000:>8.2f} | {quality_str}")
        
        # Method characteristics analysis
        print("\n" + "="*80)
        print("METHOD CHARACTERISTICS ANALYSIS")
        print("="*80)
        
        print("\n1. Computational Performance:")
        avg_times = {}
        for method in ['EDF', 'ML']:
            times = [comparison_results[inst][method]['time'] * 1000 for inst in comparison_results]
            avg_times[method] = np.mean(times)
            print(f"   {method}: Average {avg_times[method]:.2f} ms")
        
        speed_factor = avg_times['ML'] / avg_times['EDF'] if avg_times['EDF'] > 0 else float('inf')
        print(f"   Speed factor (ML/EDF): {speed_factor:.1f}x")
        
        print("\n2. Solution Quality:")
        avg_deviations = {}
        for method in ['EDF', 'ML']:
            deviations = [comparison_results[inst][method]['weighted_deviation'] for inst in comparison_results]
            avg_deviations[method] = np.mean(deviations)
            print(f"   {method}: Average weighted deviation {avg_deviations[method]:.2f}")
        
        if avg_deviations['EDF'] > 0:
            quality_improvement = (avg_deviations['EDF'] - avg_deviations['ML']) / avg_deviations['EDF'] * 100
            print(f"   ML quality improvement: {quality_improvement:+.1f}%")
        
        print("\n3. Practical Recommendations:")
        if speed_factor < 10:
            print("   ✓ ML computational overhead is acceptable")
        else:
            print("   ⚠ ML computational overhead may be concerning for real-time use")
        
        if 'ML' in comparison_results and 'avg_quality' in list(comparison_results.values())[0]['ML']:
            avg_ml_quality = np.mean([comparison_results[inst]['ML']['avg_quality'] for inst in comparison_results])
            if avg_ml_quality > 0.7:
                print("   ✓ ML provides high-quality predictions")
            elif avg_ml_quality > 0.5:
                print("   ⚠ ML provides moderate quality predictions")
            else:
                print("   ✗ ML prediction quality needs improvement")
        
        print("\n4. When to Use Each Method:")
        print("   EDF:")
        print("     - Real-time operations with minimal computational budget")
        print("     - Simple problems with clear priority structures")
        print("     - Systems with limited data for ML training")
        print("   ")
        print("   ML:")
        print("     - Complex environments with many influencing factors")
        print("     - Systems with rich historical data")
        print("     - Applications where quality predictions are valuable")
        print("     - Scenarios requiring adaptability to changing conditions")
        
        return comparison_results
    
    method_comparison = compare_ml_with_previous_methods()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ml_system' is not defined...]

### Why this Tier exists vs earlier Tiers
This Tier 4 provides machine learning capabilities that address limitations of traditional optimization methods:

- **Pattern recognition**: Learns complex relationships from historical data
- **Adaptive predictions**: Adapts to changing system conditions and patterns
- **Quality estimation**: Provides quality scores for assignment decisions
- **Feature insights**: Reveals key factors influencing appointment success
- **Generalization**: Can handle new scenarios not seen during training

### Pros / Cons vs earlier Tiers
**Pros vs Tier 1 (MIP):**
- Handles complex, non-linear relationships
- No need for explicit mathematical formulation
- Can incorporate many diverse features easily
- Provides explainable insights through feature importance

**Pros vs Tier 2 (EDF):**
- Considers multiple factors beyond just preferred time
- Learns optimal patterns from historical data
- Provides quality predictions for decision support
- More sophisticated assignment logic

**Pros vs Tier 3 (ACO):**
- Faster prediction once trained
- Provides interpretable feature insights
- Can handle very large instances efficiently
- No parameter tuning required for each problem

**Cons:**
- Requires training data and offline training phase
- Performance depends on data quality and quantity
- May not capture all problem constraints perfectly
- Model maintenance and retraining overhead
- Black-box nature (though Random Forest provides some interpretability)

### When to use this Tier
- **Data-rich environments** with historical appointment patterns
- **Complex decision environments** with many influencing factors
- **Systems requiring quality predictions** for decision support
- **Adaptive environments** where patterns change over time
- **Applications where explainability** of factors is important
- **Production systems** where model can be trained offline and deployed for fast predictions